# Gabarito — Cenário 1: Worm na Rede (CSBANK)
---
Objetivo: ler um PCAP e detectar atividade suspeita via porta 445 (SMB) usando Machine Learning (IsolationForest).

In [ ]:
# Etapa 1 — Importar bibliotecas necessárias
from scapy.all import rdpcap, IP, TCP
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
import matplotlib.pyplot as plt

In [ ]:
# Etapa 2 — Ler o arquivo PCAP
pcap_file = 'csbank_traffic.pcap'  # certifique-se que o arquivo está no mesmo diretório
packets = rdpcap(pcap_file)
print(f'Total de pacotes capturados: {len(packets)}')

In [ ]:
# Etapa 3 — Extrair estatísticas por IP de origem
from collections import defaultdict
data = defaultdict(lambda: {'pkt_count':0,'byte_count':0,'unique_dst':set(),'pkt_445':0})

for p in packets:
    if p.haslayer(IP) and p.haslayer(TCP):
        src = p[IP].src
        dst = p[IP].dst
        dport = p[TCP].dport
        data[src]['pkt_count'] += 1
        data[src]['byte_count'] += len(p)
        data[src]['unique_dst'].add(dst)
        if dport == 445:
            data[src]['pkt_445'] += 1

# Converter para DataFrame
rows = []
for ip,vals in data.items():
    rows.append({
        'src_ip': ip,
        'pkt_count': vals['pkt_count'],
        'byte_count': vals['byte_count'],
        'unique_dst': len(vals['unique_dst']),
        'pkt_445': vals['pkt_445']
    })
df = pd.DataFrame(rows)
print(df.head())

In [ ]:
# Etapa 4 — Normalizar dados e aplicar IsolationForest
features = ['pkt_count','byte_count','unique_dst','pkt_445']
X = df[features].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(n_estimators=150, contamination=0.05, random_state=42)
iso.fit(X_scaled)
scores = -iso.decision_function(X_scaled)
df['anomaly_score'] = scores
df = df.sort_values('anomaly_score', ascending=False)
df.head(10)

In [ ]:
# Etapa 5 — Exibir IPs mais suspeitos e possível paciente zero
suspect = df.iloc[0]['src_ip']
print(f'Possível paciente zero: {suspect}')
print(df.head())

In [ ]:
# Etapa 6 — Gráfico simples dos IPs mais suspeitos
top = df.head(10)
plt.figure(figsize=(8,4))
plt.barh(top['src_ip'], top['anomaly_score'], color='tomato')
plt.xlabel('Anomaly Score (maior = mais suspeito)')
plt.title('Top 10 IPs suspeitos — Worm 445')
plt.gca().invert_yaxis()
plt.show()

## Conclusão
O IP listado como *paciente zero* é aquele que iniciou o maior número de conexões na porta 445 e obteve a maior pontuação de anomalia.
Em um cenário real, recomenda-se correlacionar com logs de endpoints, firewall e antivírus antes de confirmar a infecção.